# Notebook 08: Shorter temporal comparison and sensitivity refits

**Study:** predicting recorded road casualty severity (KSI) in Great Britain. This notebook carries out the refit-based checks in section 3.7.5. The main model, settings and thresholds from notebook 05 are **not touched**. Everything here is a check on them.

**Part A. Shorter temporal comparison (develop on 2023, test on 2024).**
- Uses the frozen settings from notebook 04 for all nine model and training combinations.
- Derives each threshold inside 2023 only, from pooled out-of-fold scores of a collision-grouped, outcome-stratified 5-fold split, using the same rule as notebook 05 (macro-F1 maximiser on a 0.05 to 0.95 grid, ties closest to 0.5 then lower). 2024 never informs a threshold.
- Scores 2024 and reports the Table 3.3 measures with the paired collision bootstrap (2,000 replicates), then compares with the 2025 test.

**Part B. Sensitivity refits on the development set, scored on 2025.**
- Variant 1: police force added as a regional predictor.
- Variant 2: number of casualties in the collision removed.
- Same folds, frozen settings, thresholds re-derived on each variant's own out-of-fold scores. Paired bootstrap on 2025 against the frozen untreated models.
- Also reports AUC and the calibration intercept inside each recording-system group, to see whether adding police force changes the group differences found in notebook 07.

**Departure from the plan (logged).** Chapter 3 describes the sensitivity analyses for all nine combinations. To keep compute feasible I run Part B for the three **untreated** models only. Notebooks 06 and 07 show training treatments do not improve ranking, and the selected model is untreated. This is recorded in `departures_log.json` and must be stated in Chapter 3 or 4.

**Running it.** The notebook saves every fit as it finishes, so a disconnect loses at most one fit: rerun the cells and finished fits are reused. Set `MAX_RUN_MINUTES` to stop cleanly between fits. `RUN_A` and `RUN_B` switch the parts on and off. Use Colab High RAM if you can.

In [1]:
# 1. Setup and checks ---------------------------------------------------------------------------------
import os, sys, json, hashlib, subprocess, time, gc, warnings, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold
from scipy.stats import rankdata, spearmanr
try:
    import imblearn, psutil
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn', 'psutil'], check=True)
    import imblearn, psutil
try:
    display
except NameError:
    display = print

RUN_A = True
RUN_B = True
QUICK_TEST = False        # True = dry run on the pilot rows, separate output folders
MAX_RUN_MINUTES = 0       # 0 = no limit; otherwise stop cleanly between fits and rerun to continue
BOOT_REPS = 2000

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/stats19_severity')
else:
    PROJECT = Path(os.environ.get('PROJECT_DIR', './stats19_severity'))
CODE = PROJECT / 'code'; MOD = PROJECT / 'outputs' / 'models'; FINAL = MOD / 'final'; SEARCH = MOD / 'search'
RES = PROJECT / 'outputs' / 'results' / ('quicktest_08' if QUICK_TEST else '')
SENS = MOD / ('sensitivity_quicktest' if QUICK_TEST else 'sensitivity')
ENV = PROJECT / 'environment'; DATA = PROJECT / 'data' / 'model_inputs'; SEALED = PROJECT / 'data' / 'sealed_2025_labels'
for p in (RES, SENS): p.mkdir(parents=True, exist_ok=True)
SEED = 20261107
N_JOBS = os.cpu_count()
T_START = time.time()
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
(ENV / 'pip_freeze_notebook08.txt').write_text(subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout)

assert (FINAL / 'unseal_log.json').exists(), 'Run notebooks 05 and 06 first.'
rec = json.loads((FINAL / 'freeze_record.json').read_text())
s06 = json.loads((PROJECT / 'outputs' / 'results' / 'results_2025_summary.json').read_text())
sys.path.insert(0, str(CODE))
import importlib, stats19_lib, stats19_eval
importlib.reload(stats19_lib); importlib.reload(stats19_eval)
from stats19_lib import *
from stats19_eval import *
assert sha(CODE / 'stats19_lib.py') == rec['input_sha256']['lib'], 'The shared modelling code changed after the freeze.'
assert sha(CODE / 'stats19_eval.py') == s06['eval_module_sha256'], 'The evaluation code changed since notebook 06.'
assert sha(DATA / 'folds_dev.csv') == rec['input_sha256']['folds_dev'], 'The folds file changed.'
assert sha(SEALED / 'labels_test_2025_SEALED.csv') == s06['labels_sha256'], 'The labels differ from the file opened in notebook 06.'
print('scikit-learn', sklearn.__version__, '| cores', N_JOBS, '| QUICK_TEST', QUICK_TEST, '| frozen code, folds and labels match their checksums.')

class OutOfTime(Exception): pass
def check_time():
    if MAX_RUN_MINUTES and (time.time() - T_START) / 60 > MAX_RUN_MINUTES: raise OutOfTime()
def elapsed(): return f'{(time.time() - T_START) / 3600:.2f} h'
def save_npy(path, a):
    tmp = path.with_suffix('.tmp.npy'); np.save(tmp, np.asarray(a, dtype=np.float32)); tmp.replace(path)

Mounted at /content/drive
scikit-learn 1.6.1 | cores 2 | QUICK_TEST False | frozen code, folds and labels match their checksums.


In [2]:
# 2. Data, frozen settings, shared helpers -----------------------------------------------------------------
dev, NUM, CAT = load_dev(PROJECT)
folds = pd.read_csv(DATA / 'folds_dev.csv')[['cas_key', 'collision_index', 'source_year']]
dev = dev.merge(folds, on='cas_key', validate='one_to_one')
strata_dev = pd.read_csv(DATA / 'eval_strata_dev.csv')[['cas_key', 'police_force', 'system_group']]
dev = dev.merge(strata_dev, on='cas_key', validate='one_to_one')
if QUICK_TEST: dev = dev[dev['pilot'] == 1]
dev = dev.reset_index(drop=True)
dev['collision_index'] = dev['collision_index'].astype(str)
y_dev = dev['ksi'].values.astype(int)
SET = {k: rec['settings'][k]['params'] for k in KINDS}
COMBOS = [(k, t) for k in KINDS for t in TREATMENTS]
THR_MAIN = rec['thresholds']
print('Development rows', len(dev), '| KSI', round(y_dev.mean(), 4), '| frozen settings:', SET)

GRID = np.round(np.arange(0.05, 0.95 + 1e-9, 0.01), 2)
def counts(y, p, thresholds):
    pos = y == 1; out = []
    for t in thresholds:
        f = p >= t
        out.append((int((f & pos).sum()), int((f & ~pos).sum()), int((~f & pos).sum()), int((~f & ~pos).sum())))
    return np.array(out)
def macro_f1(c):
    tp, fp, fn, tn = c[..., 0], c[..., 1], c[..., 2], c[..., 3]
    fp_ = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
    fn_ = np.where(2 * tn + fn + fp > 0, 2 * tn / np.maximum(2 * tn + fn + fp, 1), 0.0)
    return (fp_ + fn_) / 2
def derive_threshold(y, p):
    m = macro_f1(counts(y, p, GRID)); best = m.max()
    cand = GRID[np.abs(m - best) <= 1e-12]
    return float(sorted(cand, key=lambda x: (abs(x - 0.5), x))[0]), float(best)

SCORE_KEYS = ['ap', 'auc', 'brier', 'cal_intercept', 'cal_slope']
DEC_KEYS = ['recall', 'precision', 'specificity', 'balanced_accuracy', 'macro_f1', 'flagged_share']
def safe_scores(prep, x, p, yy, w, start=(0.0, 1.0)):
    nan = {k: np.nan for k in SCORE_KEYS}
    if (w * yy).sum() <= 0 or (w * (1 - yy)).sum() <= 0: return nan, start
    try:
        with np.errstate(all='ignore'): m, warm = score_metrics(prep, x, p, yy, w, start=start)
    except (np.linalg.LinAlgError, FloatingPointError, ZeroDivisionError):
        return nan, start
    return {k: (v if np.isfinite(v) else np.nan) for k, v in m.items()}, (warm if all(np.isfinite(warm)) else start)

def boot_eval(y, codes, n_coll, P, THR, reps, seed, groups=None, label=''):
    """Paired collision bootstrap. P: name -> scores; THR: name -> {'ref', 'opt'}; groups: name -> boolean mask.
    Every series sees the same collision draws. Returns point estimates and a table of draws."""
    names = list(P)
    X = {n: clip_logit(P[n]) for n in names}
    PREP = {n: RankPrep(P[n], y) for n in names}
    DEC = [(n, r) for n in names for r in ('ref', 'opt')]
    CC = np.stack([collision_counts(codes, n_coll, (P[n] >= THR[n][r]).astype(int), y) for n, r in DEC], axis=1)
    CCf = CC.reshape(n_coll, -1)
    ones = np.ones(len(y)); point, warm = {}, {}
    for n in names:
        m, warm[n] = safe_scores(PREP[n], X[n], P[n], y.astype(float), ones)
        for k in SCORE_KEYS: point[f'{n}|{k}'] = m[k]
    with np.errstate(all='ignore'): dp = decision_rates(CC.sum(axis=0))
    for j, (n, r) in enumerate(DEC):
        for k in DEC_KEYS: point[f'{n}__{r}|{k}'] = float(dp[k][j])
    ycnt = np.bincount(codes, weights=y, minlength=n_coll); ncnt = np.bincount(codes, minlength=n_coll).astype(float)
    point['prevalence'] = float(y.mean())
    GI = {}
    if groups:
        for g, mask in groups.items():
            idx = np.flatnonzero(mask); yy = y[idx].astype(float)
            GI[g] = dict(idx=idx, y=yy, codes=codes[idx], prep={n: RankPrep(P[n][idx], yy) for n in names}, x={n: X[n][idx] for n in names})
            for n in names:
                with np.errstate(all='ignore'):
                    ap_, auc_ = GI[g]['prep'][n].ap_auc(np.ones(len(idx)))
                    ci_, _ = irls_logistic(GI[g]['x'][n], yy, None, fit_slope=False)
                point[f'{n}|{g}|auc'] = auc_; point[f'{n}|{g}|cal_intercept'] = ci_
    draws = {k: np.full(reps, np.nan) for k in point}
    rng = np.random.RandomState(seed); t0 = time.time()
    for b in range(reps):
        wc = rng.multinomial(n_coll, np.full(n_coll, 1.0 / n_coll)).astype(float)
        wr = wc[codes]
        draws['prevalence'][b] = (wc @ ycnt) / (wc @ ncnt)
        for n in names:
            m, _ = safe_scores(PREP[n], X[n], P[n], y.astype(float), wr, start=warm[n])
            for k in SCORE_KEYS: draws[f'{n}|{k}'][b] = m[k]
        with np.errstate(all='ignore'): r_ = decision_rates((wc @ CCf).reshape(len(DEC), 4))
        for j, (n, r) in enumerate(DEC):
            for k in DEC_KEYS: draws[f'{n}__{r}|{k}'][b] = r_[k][j]
        for g, gi in GI.items():
            w = wc[gi['codes']]
            if (w * gi['y']).sum() <= 0 or (w * (1 - gi['y'])).sum() <= 0: continue
            for n in names:
                try:
                    with np.errstate(all='ignore'):
                        _, auc_ = gi['prep'][n].ap_auc(w); ci_, _ = irls_logistic(gi['x'][n], gi['y'], w, fit_slope=False)
                except np.linalg.LinAlgError:
                    continue
                draws[f'{n}|{g}|auc'][b] = auc_ if np.isfinite(auc_) else np.nan
                draws[f'{n}|{g}|cal_intercept'][b] = ci_ if np.isfinite(ci_) else np.nan
        if (b + 1) % 250 == 0: print(f'  {label} bootstrap {b + 1}/{reps} | {(time.time() - t0) / 60:.1f} min')
    return point, pd.DataFrame(draws)

def ci(boot, key, reps):
    v = boot[key].values; v = v[np.isfinite(v)]
    return tuple(np.percentile(v, [2.5, 97.5])) if len(v) >= 0.95 * reps else (np.nan, np.nan)
def contrast(point, boot, ka, kb, reps):
    d = boot[ka].values - boot[kb].values; d = d[np.isfinite(d)]
    lo, hi = np.percentile(d, [2.5, 97.5]) if len(d) >= 0.95 * reps else (np.nan, np.nan)
    return dict(a=ka, b=kb, difference=point[ka] - point[kb], lo=lo, hi=hi, interval_excludes_zero=bool(np.isfinite(lo) and (lo > 0 or hi < 0)))
fmt = lambda e, lo, hi, d=3: f'{e:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]' if np.isfinite(e) and np.isfinite(lo) else 'not estimable'
REPS = 40 if QUICK_TEST else BOOT_REPS

Development rows 261249 | KSI 0.2117 | frozen settings: {'lr': {'C': 0.1}, 'rf': {'max_features': 0.2, 'min_samples_leaf': 10, 'min_samples_split': 10}, 'hgb': {'learning_rate': 0.05, 'max_iter': 200, 'max_leaf_nodes': 63, 'min_samples_leaf': 50, 'l2_regularization': 0.0}}


In [3]:
# 3. Part A: develop on 2023, test on 2024 -- fits ------------------------------------------------------------------------
if RUN_A:
    d23 = dev[dev['source_year'] == 2023].reset_index(drop=True); d24 = dev[dev['source_year'] == 2024].reset_index(drop=True)
    y23, y24 = d23['ksi'].values.astype(int), d24['ksi'].values.astype(int)
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + 5)
    fold23 = np.full(len(d23), -1)
    for j, (_, va) in enumerate(sgkf.split(d23, y23, groups=d23['collision_index'].values)): fold23[va] = j
    assert (fold23 >= 0).all()
    pd.DataFrame({'cas_key': d23['cas_key'], 'fold': fold23}).to_csv(SENS / 'folds_2023_only.csv', index=False)
    print(f'2023 rows {len(d23)} (KSI {y23.mean():.4f}) | 2024 rows {len(d24)} (KSI {y24.mean():.4f}) | folds sha256 {sha(SENS / "folds_2023_only.csv")[:16]}')
    oof23 = {}; p24 = {}
    try:
        for kind, trt in COMBOS:
            f_final = SENS / f'A_2024__{kind}__{trt}.npy'
            o = np.zeros(len(d23))
            for j in range(5):
                f = SENS / f'A_cv2023__{kind}__{trt}__fold{j}.npy'
                va_idx = np.flatnonzero(fold23 == j)
                if f.exists(): o[va_idx] = np.load(f); continue
                check_time()
                tr = d23[fold23 != j]; va = d23.iloc[va_idx]
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    p, info = fit_predict(kind, SET[kind], trt, tr, tr['ksi'], va, NUM, CAT, SEED, N_JOBS)
                save_npy(f, p); o[va_idx] = p
                print(f"A cv {kind:>3} {trt:<10} fold {j} | {(info['t_prep'] + info['t_fit'] + info['t_pred']) / 60:.1f} min | elapsed {elapsed()}")
                gc.collect()
            oof23[(kind, trt)] = o
            if f_final.exists(): p24[(kind, trt)] = np.load(f_final); continue
            check_time()
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                p, info = fit_predict(kind, SET[kind], trt, d23, d23['ksi'], d24, NUM, CAT, SEED, N_JOBS)
            save_npy(f_final, p); p24[(kind, trt)] = np.load(f_final)
            print(f"A final {kind:>3} {trt:<10} trained on 2023 ({info['n_train']} rows), scored 2024 | {(info['t_prep'] + info['t_fit'] + info['t_pred']) / 60:.1f} min | elapsed {elapsed()}")
            gc.collect()
    except OutOfTime:
        print(f'Stopped after {MAX_RUN_MINUTES} minutes. Rerun this cell to continue; finished fits are kept.')
        raise SystemExit
    THR_A = {}
    rows = []
    for c in COMBOS:
        opt, best = derive_threshold(y23, oof23[c])
        THR_A[c] = dict(ref=0.5, opt=opt)
        rows.append(dict(model=c[0], training=c[1], threshold_2023=opt, oof_macro_f1_2023=best, frozen_threshold_2023_24=THR_MAIN[f'{c[0]}__{c[1]}']['opt']))
    thr_a = pd.DataFrame(rows); thr_a.round(4).to_csv(RES / 'temporal_2023_to_2024_thresholds.csv', index=False)
    print('All 2023 fits finished. Thresholds derived inside 2023 only:'); display(thr_a.round(3))

2023 rows 132977 (KSI 0.2080) | 2024 rows 128272 (KSI 0.2155) | folds sha256 f895e27cdf64a5eb
A cv  lr untreated  fold 0 | 0.1 min | elapsed 0.02 h
A cv  lr untreated  fold 1 | 0.0 min | elapsed 0.02 h
A cv  lr untreated  fold 2 | 0.1 min | elapsed 0.02 h
A cv  lr untreated  fold 3 | 0.0 min | elapsed 0.02 h
A cv  lr untreated  fold 4 | 0.0 min | elapsed 0.02 h
A final  lr untreated  trained on 2023 (132977 rows), scored 2024 | 0.1 min | elapsed 0.03 h
A cv  lr weight     fold 0 | 0.1 min | elapsed 0.03 h
A cv  lr weight     fold 1 | 0.0 min | elapsed 0.03 h
A cv  lr weight     fold 2 | 0.1 min | elapsed 0.03 h
A cv  lr weight     fold 3 | 0.1 min | elapsed 0.03 h
A cv  lr weight     fold 4 | 0.1 min | elapsed 0.03 h
A final  lr weight     trained on 2023 (132977 rows), scored 2024 | 0.1 min | elapsed 0.03 h
A cv  lr oversample fold 0 | 0.1 min | elapsed 0.03 h
A cv  lr oversample fold 1 | 0.1 min | elapsed 0.04 h
A cv  lr oversample fold 2 | 0.1 min | elapsed 0.04 h
A cv  lr oversampl

,model,training,threshold_2023,oof_macro_f1_2023,frozen_threshold_2023_24
0,lr,untreated,0.30,0.642,0.31
1,lr,weight,0.62,0.641,0.62
2,lr,oversample,0.63,0.641,0.62
3,rf,untreated,0.30,0.648,0.30
4,rf,weight,0.53,0.647,0.52
5,rf,oversample,0.53,0.647,0.51
6,hgb,untreated,0.30,0.651,0.30
7,hgb,weight,0.62,0.651,0.62
8,hgb,oversample,0.61,0.650,0.62


In [4]:
# 4. Part A: evaluate on 2024 with the paired collision bootstrap, compare with the 2025 test -----------------------------
if RUN_A:
    codes24, idx24 = pd.factorize(d24['collision_index']); n24 = len(idx24)
    PA = {f'{k}__{t}': p24[(k, t)].astype(float) for k, t in COMBOS}
    TA = {f'{k}__{t}': THR_A[(k, t)] for k, t in COMBOS}
    pointA, bootA = boot_eval(y24, codes24, n24, PA, TA, REPS, SEED + 6, label='2024')
    bootA.to_parquet(RES / 'bootstrap_draws_temporal_2024.parquet', index=False)
    rows = []
    for n in PA:
        k, t = n.split('__')
        base = dict(model=k, training=t, threshold=TA[n]['opt'], observed_prevalence=pointA['prevalence'], mean_score=PA[n].mean())
        for m in SCORE_KEYS:
            lo, hi = ci(bootA, f'{n}|{m}', REPS); base.update({m: pointA[f'{n}|{m}'], m + '_lo': lo, m + '_hi': hi})
        rows.append(base)
    resA = pd.DataFrame(rows); resA.round(5).to_csv(RES / 'temporal_2023_to_2024_score_measures.csv', index=False)
    rows = []
    for n in PA:
        k, t = n.split('__')
        for r in ('ref', 'opt'):
            base = dict(model=k, training=t, rule=r, threshold=TA[n][r])
            for m in DEC_KEYS:
                lo, hi = ci(bootA, f'{n}__{r}|{m}', REPS); base.update({m: pointA[f'{n}__{r}|{m}'], m + '_lo': lo, m + '_hi': hi})
            rows.append(base)
    decA = pd.DataFrame(rows); decA.round(5).to_csv(RES / 'temporal_2023_to_2024_decision_measures.csv', index=False)
    show = resA.copy()
    for m in ('ap', 'auc', 'brier', 'cal_intercept', 'cal_slope'): show[m] = [fmt(a, b, c) for a, b, c in zip(resA[m], resA[m + '_lo'], resA[m + '_hi'])]
    print('Trained on 2023, tested on 2024 (estimate [95% interval]):'); display(show[['model', 'training', 'threshold', 'ap', 'auc', 'brier', 'cal_intercept', 'cal_slope']])
    sd = decA[decA.rule == 'opt'].copy()
    for m in ('recall', 'precision', 'specificity', 'macro_f1', 'flagged_share'): sd[m] = [fmt(a, b, c) for a, b, c in zip(sd[m], sd[m + '_lo'], sd[m + '_hi'])]
    print('\nDecision measures at the thresholds derived inside 2023:'); display(sd[['model', 'training', 'threshold', 'recall', 'precision', 'specificity', 'macro_f1', 'flagged_share']])

    cons = []
    for a, b in [('hgb__untreated', 'rf__untreated'), ('hgb__untreated', 'lr__untreated'), ('rf__untreated', 'lr__untreated')]:
        for m in ('ap', 'auc', 'brier'): cons.append(contrast(pointA, bootA, f'{a}|{m}', f'{b}|{m}', REPS))
        cons.append(contrast(pointA, bootA, f'{a}__opt|macro_f1', f'{b}__opt|macro_f1', REPS))
    for k in KINDS:
        for t in ('weight', 'oversample'):
            for m in ('ap', 'brier', 'cal_intercept'): cons.append(contrast(pointA, bootA, f'{k}__{t}|{m}', f'{k}__untreated|{m}', REPS))
            for r in ('opt', 'ref'): cons.append(contrast(pointA, bootA, f'{k}__{t}__{r}|macro_f1', f'{k}__untreated__{r}|macro_f1', REPS))
    consA = pd.DataFrame(cons); consA.round(5).to_csv(RES / 'temporal_2023_to_2024_paired_contrasts.csv', index=False)
    print('\nPaired differences on 2024 (first minus second):'); display(consA.round(4))

    sc25 = pd.read_csv(PROJECT / 'outputs' / 'results' / 'results_2025_score_measures.csv'); dc25 = pd.read_csv(PROJECT / 'outputs' / 'results' / 'results_2025_decisions.csv')
    rows = []
    for k, t in COMBOS:
        r24 = resA[(resA.model == k) & (resA.training == t)].iloc[0]; d24_ = decA[(decA.model == k) & (decA.training == t) & (decA.rule == 'opt')].iloc[0]
        e = lambda m: sc25[(sc25.model == k) & (sc25.training == t) & (sc25.measure == m)]['estimate'].iloc[0]
        d25 = dc25[(dc25.model == k) & (dc25.training == t) & (dc25.rule == 'opt')].iloc[0]
        rows.append(dict(model=k, training=t, auc_2024_from_2023=r24['auc'], auc_2025_from_2023_24=e('auc'), ap_2024_from_2023=r24['ap'], ap_2025_from_2023_24=e('ap'),
                         brier_2024=r24['brier'], brier_2025=e('brier'), cal_intercept_2024=r24['cal_intercept'], cal_intercept_2025=e('cal_intercept'),
                         macro_f1_opt_2024=d24_['macro_f1'], macro_f1_opt_2025=d25['macro_f1'], recall_opt_2024=d24_['recall'], recall_opt_2025=d25['recall']))
    cmp = pd.DataFrame(rows)
    cmp.round(4).to_csv(RES / 'temporal_2023_to_2024_vs_2025_test.csv', index=False)
    print(f'\nSide by side with the main 2025 test (trained on {len(dev)} rows in 2023-24, versus {len(d23)} rows in 2023 here):'); display(cmp.round(4))
    ag = []
    for lab, a, b in (('ROC-AUC', 'auc_2024_from_2023', 'auc_2025_from_2023_24'), ('average precision', 'ap_2024_from_2023', 'ap_2025_from_2023_24'), ('macro-F1 at tuned threshold', 'macro_f1_opt_2024', 'macro_f1_opt_2025')):
        ag.append(dict(measure=lab, spearman_nine_combinations=spearmanr(cmp[a], cmp[b])[0], best_2024=cmp.loc[cmp[a].idxmax(), 'model'] + '/' + cmp.loc[cmp[a].idxmax(), 'training'], best_2025=cmp.loc[cmp[b].idxmax(), 'model'] + '/' + cmp.loc[cmp[b].idxmax(), 'training']))
    ag = pd.DataFrame(ag); ag.round(3).to_csv(RES / 'temporal_2023_to_2024_ranking_agreement.csv', index=False)
    print('\nDoes the ordering of the nine combinations agree between the two temporal tests?'); display(ag.round(3))
    print('The 2023 to 2024 test trains on about half the data and tests one year ahead, so lower or different absolute values are expected. The comparison is about ordering and calibration drift, not equal scores.')

  2024 bootstrap 250/2000 | 0.9 min
  2024 bootstrap 500/2000 | 1.7 min
  2024 bootstrap 750/2000 | 2.5 min
  2024 bootstrap 1000/2000 | 3.4 min
  2024 bootstrap 1250/2000 | 4.2 min
  2024 bootstrap 1500/2000 | 5.0 min
  2024 bootstrap 1750/2000 | 5.9 min
  2024 bootstrap 2000/2000 | 6.7 min
Trained on 2023, tested on 2024 (estimate [95% interval]):


,model,training,threshold,ap,auc,brier,cal_intercept,cal_slope
0,lr,untreated,0.30,"0.428 [0.422, 0.435]","0.728 [0.725, 0.732]","0.149 [0.148, 0.151]","0.073 [0.057, 0.089]","0.916 [0.899, 0.935]"
1,lr,weight,0.62,"0.426 [0.420, 0.433]","0.728 [0.724, 0.731]","0.207 [0.206, 0.208]","-1.253 [-1.269, -1.237]","0.909 [0.892, 0.928]"
2,lr,oversample,0.63,"0.426 [0.420, 0.432]","0.728 [0.724, 0.731]","0.209 [0.208, 0.210]","-1.274 [-1.290, -1.258]","0.902 [0.885, 0.921]"
3,rf,untreated,0.30,"0.441 [0.435, 0.448]","0.735 [0.732, 0.738]","0.148 [0.146, 0.149]","0.054 [0.039, 0.070]","1.022 [1.003, 1.042]"
4,rf,weight,0.53,"0.438 [0.432, 0.444]","0.733 [0.729, 0.736]","0.175 [0.174, 0.176]","-0.874 [-0.889, -0.858]","0.993 [0.975, 1.012]"
5,rf,oversample,0.53,"0.436 [0.430, 0.443]","0.733 [0.729, 0.736]","0.174 [0.173, 0.175]","-0.860 [-0.875, -0.844]","0.998 [0.980, 1.017]"
6,hgb,untreated,0.30,"0.447 [0.440, 0.453]","0.738 [0.735, 0.741]","0.147 [0.146, 0.148]","0.087 [0.072, 0.103]","0.960 [0.944, 0.978]"
7,hgb,weight,0.62,"0.445 [0.439, 0.452]","0.739 [0.735, 0.742]","0.199 [0.198, 0.200]","-1.191 [-1.207, -1.175]","0.963 [0.946, 0.982]"
8,hgb,oversample,0.61,"0.445 [0.438, 0.451]","0.737 [0.734, 0.740]","0.199 [0.198, 0.200]","-1.188 [-1.203, -1.172]","0.963 [0.945, 0.982]"



Decision measures at the thresholds derived inside 2023:


,model,training,threshold,recall,precision,specificity,macro_f1,flagged_share
1,lr,untreated,0.30,"0.445 [0.439, 0.451]","0.422 [0.416, 0.428]","0.833 [0.830, 0.835]","0.636 [0.633, 0.639]","0.227 [0.224, 0.230]"
3,lr,weight,0.62,"0.439 [0.433, 0.445]","0.423 [0.417, 0.429]","0.836 [0.833, 0.838]","0.635 [0.632, 0.639]","0.224 [0.221, 0.226]"
5,lr,oversample,0.63,"0.430 [0.424, 0.437]","0.426 [0.420, 0.432]","0.841 [0.838, 0.843]","0.635 [0.632, 0.638]","0.218 [0.215, 0.220]"
7,rf,untreated,0.30,"0.444 [0.438, 0.451]","0.436 [0.430, 0.442]","0.842 [0.840, 0.845]","0.642 [0.639, 0.645]","0.220 [0.217, 0.222]"
9,rf,weight,0.53,"0.422 [0.416, 0.428]","0.438 [0.432, 0.444]","0.851 [0.849, 0.853]","0.638 [0.635, 0.642]","0.208 [0.205, 0.210]"
11,rf,oversample,0.53,"0.412 [0.405, 0.418]","0.443 [0.437, 0.449]","0.858 [0.855, 0.860]","0.638 [0.635, 0.641]","0.200 [0.198, 0.203]"
13,hgb,untreated,0.30,"0.445 [0.439, 0.451]","0.441 [0.435, 0.447]","0.845 [0.843, 0.847]","0.644 [0.641, 0.648]","0.217 [0.215, 0.220]"
15,hgb,weight,0.62,"0.424 [0.418, 0.431]","0.446 [0.440, 0.452]","0.855 [0.853, 0.857]","0.642 [0.639, 0.645]","0.205 [0.203, 0.208]"
17,hgb,oversample,0.61,"0.445 [0.439, 0.452]","0.435 [0.429, 0.441]","0.841 [0.839, 0.843]","0.642 [0.639, 0.645]","0.221 [0.218, 0.223]"



Paired differences on 2024 (first minus second):


,a,b,difference,lo,hi,interval_excludes_zero
0,hgb__untreated|ap,rf__untreated|ap,0.0054,0.0031,0.0077,True
1,hgb__untreated|auc,rf__untreated|auc,0.0030,0.0017,0.0044,True
2,hgb__untreated|brier,rf__untreated|brier,-0.0006,-0.0009,-0.0004,True
3,hgb__untreated__opt|macro_f1,rf__untreated__opt|macro_f1,0.0021,-0.0000,0.0044,False
4,hgb__untreated|ap,lr__untreated|ap,0.0181,0.0153,0.0214,True
5,hgb__untreated|auc,lr__untreated|auc,0.0099,0.0084,0.0115,True
6,hgb__untreated|brier,lr__untreated|brier,-0.0021,-0.0025,-0.0018,True
7,hgb__untreated__opt|macro_f1,lr__untreated__opt|macro_f1,0.0082,0.0057,0.0107,True
8,rf__untreated|ap,lr__untreated|ap,0.0127,0.0093,0.0161,True
9,rf__untreated|auc,lr__untreated|auc,0.0069,0.0051,0.0088,True



Side by side with the main 2025 test (trained on 261249 rows in 2023-24, versus 132977 rows in 2023 here):


,model,training,auc_2024_from_2023,auc_2025_from_2023_24,ap_2024_from_2023,ap_2025_from_2023_24,brier_2024,brier_2025,cal_intercept_2024,cal_intercept_2025,macro_f1_opt_2024,macro_f1_opt_2025,recall_opt_2024,recall_opt_2025
0,lr,untreated,0.7281,0.7236,0.4285,0.4418,0.1493,0.1559,0.0730,0.0524,0.6362,0.6390,0.4450,0.4631
1,lr,weight,0.7277,0.7232,0.4264,0.4383,0.2069,0.2171,-1.2534,-1.2643,0.6354,0.6377,0.4392,0.4740
2,lr,oversample,0.7275,0.7232,0.4259,0.4382,0.2090,0.2174,-1.2742,-1.2674,0.6350,0.6377,0.4302,0.4757
3,rf,untreated,0.7350,0.7335,0.4412,0.4592,0.1478,0.1537,0.0541,0.0355,0.6422,0.6453,0.4444,0.4931
4,rf,weight,0.7328,0.7340,0.4378,0.4598,0.1747,0.1818,-0.8740,-0.8847,0.6385,0.6463,0.4222,0.4888
5,rf,oversample,0.7326,0.7325,0.4365,0.4579,0.1739,0.1810,-0.8599,-0.8675,0.6380,0.6447,0.4117,0.4967
6,hgb,untreated,0.7380,0.7376,0.4466,0.4656,0.1471,0.1528,0.0874,0.0484,0.6444,0.6494,0.4446,0.4983
7,hgb,weight,0.7387,0.7374,0.4451,0.4639,0.1989,0.2102,-1.1910,-1.2264,0.6421,0.6482,0.4243,0.4746
8,hgb,oversample,0.7370,0.7369,0.4445,0.4634,0.1990,0.2099,-1.1875,-1.2207,0.6418,0.6479,0.4450,0.4721



Does the ordering of the nine combinations agree between the two temporal tests?


,measure,spearman_nine_combinations,best_2024,best_2025
0,ROC-AUC,0.967,hgb/weight,hgb/untreated
1,average precision,0.983,hgb/untreated,hgb/untreated
2,macro-F1 at tuned threshold,0.883,hgb/untreated,hgb/untreated


The 2023 to 2024 test trains on about half the data and tests one year ahead, so lower or different absolute values are expected. The comparison is about ordering and calibration drift, not equal scores.


In [5]:
# 5. Part B: sensitivity refits on the development set (untreated models) ---------------------------------------------------
VARIANTS = {'force_included': dict(num=NUM, cat=CAT + ['police_force']), 'casualty_count_removed': dict(num=[c for c in NUM if c != 'number_of_casualties'], cat=CAT)}
assert 'number_of_casualties' in NUM, 'number_of_casualties should be a primary numeric field'
UNT = [(k, 'untreated') for k in KINDS]
labels = pd.read_csv(SEALED / 'labels_test_2025_SEALED.csv')
scores = pd.read_parquet(FINAL / rec['scores_2025']['file'])
assert sha(FINAL / rec['scores_2025']['file']) == rec['scores_2025']['sha256']
strata25 = pd.read_csv(DATA / 'eval_strata_2025.csv')
df = scores.merge(labels, on='cas_key', validate='one_to_one').merge(strata25, on='cas_key', validate='one_to_one')
df['ksi'] = df['ksi'].astype(int); df['coll'] = df['cas_key'].str.split('|').str[0]
test = pd.read_parquet(DATA / 'features_test_2025_NO_LABELS.parquet') if (DATA / 'features_test_2025_NO_LABELS.parquet').exists() else pd.read_pickle(DATA / 'features_test_2025_NO_LABELS.pkl')
test = test.set_index('cas_key').loc[df['cas_key']].reset_index()
test['police_force'] = df['police_force'].values
y25 = df['ksi'].values
print('2025 rows scored in Part B:', len(test), '| KSI', y25.sum())

if RUN_B:
    oof_path = next((p for p in (SEARCH / 'oof_scores_dev.parquet', SEARCH / 'oof_scores_dev.pkl') if p.exists()), None)
    oof_main = (pd.read_parquet(oof_path) if oof_path.suffix == '.parquet' else pd.read_pickle(oof_path)) if oof_path else None
    if not QUICK_TEST:                                   # the helper must reproduce the frozen thresholds from the saved out-of-fold scores
        assert (oof_main['cas_key'].values == dev['cas_key'].values).all()
        for k, t in UNT:
            assert abs(derive_threshold(y_dev, oof_main[f'{k}__{t}'].values)[0] - THR_MAIN[f'{k}__{t}']['opt']) < 1e-9, 'Threshold helper disagrees with the frozen threshold.'
        print('Threshold helper reproduces the three frozen untreated thresholds.')
    oofB, p25B, thrB = {}, {}, []
    try:
        for vname, v in VARIANTS.items():
            for kind, trt in UNT:
                o = np.zeros(len(dev))
                for j in range(5):
                    f = SENS / f'B_{vname}__{kind}__fold{j}.npy'
                    va_idx = np.flatnonzero(dev['fold'].values == j)
                    if f.exists(): o[va_idx] = np.load(f); continue
                    check_time()
                    tr = dev[dev['fold'] != j]; va = dev.iloc[va_idx]
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore')
                        p, info = fit_predict(kind, SET[kind], trt, tr, tr['ksi'], va, v['num'], v['cat'], SEED, N_JOBS)
                    save_npy(f, p); o[va_idx] = p
                    print(f"B {vname} cv {kind:>3} fold {j} | {(info['t_prep'] + info['t_fit'] + info['t_pred']) / 60:.1f} min | elapsed {elapsed()}")
                    gc.collect()
                oofB[(vname, kind)] = o
                f = SENS / f'B_{vname}__{kind}__2025.npy'
                if not f.exists():
                    check_time()
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore')
                        p, info = fit_predict(kind, SET[kind], trt, dev, dev['ksi'], test, v['num'], v['cat'], SEED, N_JOBS)
                    assert len(p) == len(test) and np.isfinite(p).all()
                    save_npy(f, p)
                    print(f"B {vname} final {kind:>3} refit on {info['n_train']} rows, scored 2025 | {(info['t_prep'] + info['t_fit'] + info['t_pred']) / 60:.1f} min | elapsed {elapsed()}")
                    gc.collect()
                p25B[(vname, kind)] = np.load(f)
    except OutOfTime:
        print(f'Stopped after {MAX_RUN_MINUTES} minutes. Rerun this cell to continue; finished fits are kept.')
        raise SystemExit
    from sklearn.metrics import average_precision_score
    THR_B = {}
    for (vname, kind), o in oofB.items():
        opt, best = derive_threshold(y_dev, o)
        THR_B[(vname, kind)] = dict(ref=0.5, opt=opt)
        base = oof_main[f'{kind}__untreated'].values if (oof_main is not None and not QUICK_TEST) else None
        thrB.append(dict(variant=vname, model=kind, threshold=opt, frozen_threshold=THR_MAIN[f'{kind}__untreated']['opt'], oof_macro_f1=best,
                         oof_ap=average_precision_score(y_dev, o), oof_ap_frozen_model=average_precision_score(y_dev, base) if base is not None else np.nan))
    thrB = pd.DataFrame(thrB); thrB.round(4).to_csv(RES / 'sensitivity_thresholds_and_dev_oof.csv', index=False)
    print('Sensitivity fits finished. Thresholds re-derived on each variant\'s own out-of-fold scores:'); display(thrB.round(4))

2025 rows scored in Part B: 127883 | KSI 29296
Threshold helper reproduces the three frozen untreated thresholds.
B force_included cv  lr fold 0 | 0.1 min | elapsed 0.94 h
B force_included cv  lr fold 1 | 0.1 min | elapsed 0.94 h
B force_included cv  lr fold 2 | 0.1 min | elapsed 0.94 h
B force_included cv  lr fold 3 | 0.1 min | elapsed 0.94 h
B force_included cv  lr fold 4 | 0.1 min | elapsed 0.95 h
B force_included final  lr refit on 261249 rows, scored 2025 | 0.1 min | elapsed 0.95 h
B force_included cv  rf fold 0 | 2.3 min | elapsed 0.99 h
B force_included cv  rf fold 1 | 2.2 min | elapsed 1.02 h
B force_included cv  rf fold 2 | 2.3 min | elapsed 1.06 h
B force_included cv  rf fold 3 | 2.2 min | elapsed 1.10 h
B force_included cv  rf fold 4 | 2.2 min | elapsed 1.14 h
B force_included final  rf refit on 261249 rows, scored 2025 | 3.0 min | elapsed 1.19 h
B force_included cv hgb fold 0 | 2.8 min | elapsed 1.23 h
B force_included cv hgb fold 1 | 2.7 min | elapsed 1.28 h
B force_includ

,variant,model,threshold,frozen_threshold,oof_macro_f1,oof_ap,oof_ap_frozen_model
0,force_included,lr,0.31,0.31,0.6468,0.4456,0.4344
1,force_included,rf,0.30,0.30,0.6503,0.4539,0.4484
2,force_included,hgb,0.30,0.30,0.6558,0.4660,0.4556
3,casualty_count_removed,lr,0.30,0.31,0.6411,0.4357,0.4344
4,casualty_count_removed,rf,0.30,0.30,0.6478,0.4476,0.4484
5,casualty_count_removed,hgb,0.31,0.30,0.6502,0.4549,0.4556


In [6]:
# 6. Part B: evaluate on 2025 against the frozen untreated models -------------------------------------------------------------
if RUN_B:
    codes25, idx25 = pd.factorize(df['coll']); n25 = len(idx25)
    PB = {f'base__{k}': df[f'{k}__untreated'].values.astype(float) for k in KINDS}
    TB = {f'base__{k}': dict(ref=0.5, opt=THR_MAIN[f'{k}__untreated']['opt']) for k in KINDS}
    for vname in VARIANTS:
        for k in KINDS:
            PB[f'{vname}__{k}'] = p25B[(vname, k)].astype(float); TB[f'{vname}__{k}'] = THR_B[(vname, k)]
    SYSG = ['stable injury-based', 'stable non-injury-based', 'changed or mixed']
    GB = {g: (df['system_group'] == g).values for g in SYSG}
    pointB, bootB = boot_eval(y25, codes25, n25, PB, TB, REPS, SEED + 7, groups=GB, label='2025 sensitivity')
    bootB.to_parquet(RES / 'bootstrap_draws_sensitivity_2025.parquet', index=False)
    rows = []
    for n in PB:
        v, k = n.split('__')
        base = dict(variant=v, model=k, threshold=TB[n]['opt'], mean_score=PB[n].mean())
        for m in SCORE_KEYS:
            lo, hi = ci(bootB, f'{n}|{m}', REPS); base.update({m: pointB[f'{n}|{m}'], m + '_lo': lo, m + '_hi': hi})
        for m in DEC_KEYS:
            lo, hi = ci(bootB, f'{n}__opt|{m}', REPS); base.update({m + '_at_opt': pointB[f'{n}__opt|{m}'], m + '_at_opt_lo': lo, m + '_at_opt_hi': hi})
        rows.append(base)
    resB = pd.DataFrame(rows); resB.round(5).to_csv(RES / 'sensitivity_2025_measures.csv', index=False)
    show = resB.copy()
    for m in ('ap', 'auc', 'cal_intercept'): show[m] = [fmt(a, b, c) for a, b, c in zip(resB[m], resB[m + '_lo'], resB[m + '_hi'])]
    show['macro_f1'] = [fmt(a, b, c) for a, b, c in zip(resB['macro_f1_at_opt'], resB['macro_f1_at_opt_lo'], resB['macro_f1_at_opt_hi'])]
    print('2025 results by variant, untreated models (baseline = frozen models from notebook 05):'); display(show[['variant', 'model', 'threshold', 'ap', 'auc', 'cal_intercept', 'macro_f1']])

    cons = []
    for vname in VARIANTS:
        for k in KINDS:
            a, b = f'{vname}__{k}', f'base__{k}'
            for m in ('ap', 'auc', 'brier', 'cal_intercept', 'cal_slope'): cons.append(dict(variant=vname, model=k, measure=m, **contrast(pointB, bootB, f'{a}|{m}', f'{b}|{m}', REPS)))
            for m in ('macro_f1', 'recall', 'precision', 'flagged_share'): cons.append(dict(variant=vname, model=k, measure=m + ' (own tuned threshold)', **contrast(pointB, bootB, f'{a}__opt|{m}', f'{b}__opt|{m}', REPS)))
    consB = pd.DataFrame(cons).drop(columns=['a', 'b']); consB.round(5).to_csv(RES / 'sensitivity_2025_paired_contrasts.csv', index=False)
    print('\nVariant minus frozen baseline, same collision draws (95% interval):')
    display(consB[consB.measure.isin(['ap', 'auc', 'cal_intercept', 'macro_f1 (own tuned threshold)'])].round(4))

    rows = []
    for n in PB:
        v, k = n.split('__')
        for g in SYSG:
            for m in ('auc', 'cal_intercept'):
                lo, hi = ci(bootB, f'{n}|{g}|{m}', REPS); rows.append(dict(variant=v, model=k, group=g, measure=m, estimate=pointB[f'{n}|{g}|{m}'], lo=lo, hi=hi))
    sysB = pd.DataFrame(rows); sysB.round(5).to_csv(RES / 'sensitivity_2025_by_recording_system.csv', index=False)
    gap = []
    for n in PB:
        v, k = n.split('__')
        d = bootB[f'{n}|stable injury-based|cal_intercept'].values - bootB[f'{n}|stable non-injury-based|cal_intercept'].values; d = d[np.isfinite(d)]
        est = pointB[f'{n}|stable injury-based|cal_intercept'] - pointB[f'{n}|stable non-injury-based|cal_intercept']
        gap.append(dict(variant=v, model=k, intercept_gap_injury_minus_non_injury=est, lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5)))
    gap = pd.DataFrame(gap); gap.round(5).to_csv(RES / 'sensitivity_2025_system_calibration_gap.csv', index=False)
    print('\nCalibration intercept gap between stable injury-based and stable non-injury-based forces (baseline gap from notebook 07 for hgb was 0.19):'); display(gap.round(4))

    rows = []
    for v in ['base'] + list(VARIANTS):
        for m in ('ap', 'auc'):
            M = np.column_stack([bootB[f'{v}__{k}|{m}'].values for k in KINDS])
            first = (M.argmax(axis=1) == KINDS.index('hgb')).mean()
            order = ' > '.join(sorted(KINDS, key=lambda k: -pointB[f'{v}__{k}|{m}']))
            rows.append(dict(variant=v, measure=m, point_order=order, share_of_draws_hgb_first=first))
    ordB = pd.DataFrame(rows); ordB.round(4).to_csv(RES / 'sensitivity_2025_model_order.csv', index=False)
    print('\nModel ordering inside each variant:'); display(ordB.round(4))

  2025 sensitivity bootstrap 250/2000 | 1.3 min
  2025 sensitivity bootstrap 500/2000 | 2.5 min
  2025 sensitivity bootstrap 750/2000 | 3.8 min
  2025 sensitivity bootstrap 1000/2000 | 5.1 min
  2025 sensitivity bootstrap 1250/2000 | 6.3 min
  2025 sensitivity bootstrap 1500/2000 | 7.6 min
  2025 sensitivity bootstrap 1750/2000 | 8.9 min
  2025 sensitivity bootstrap 2000/2000 | 10.2 min
2025 results by variant, untreated models (baseline = frozen models from notebook 05):


,variant,model,threshold,ap,auc,cal_intercept,macro_f1
0,base,lr,0.31,"0.442 [0.433, 0.450]","0.724 [0.720, 0.727]","0.052 [0.036, 0.068]","0.639 [0.636, 0.642]"
1,base,rf,0.30,"0.459 [0.453, 0.466]","0.733 [0.730, 0.737]","0.036 [0.021, 0.050]","0.645 [0.642, 0.648]"
2,base,hgb,0.30,"0.466 [0.460, 0.472]","0.738 [0.734, 0.741]","0.048 [0.034, 0.063]","0.649 [0.646, 0.653]"
3,force_included,lr,0.31,"0.445 [0.433, 0.456]","0.728 [0.724, 0.732]","0.097 [0.079, 0.114]","0.642 [0.639, 0.645]"
4,force_included,rf,0.30,"0.463 [0.457, 0.469]","0.737 [0.733, 0.740]","0.043 [0.029, 0.058]","0.647 [0.644, 0.650]"
5,force_included,hgb,0.30,"0.472 [0.466, 0.479]","0.742 [0.739, 0.746]","0.078 [0.064, 0.093]","0.653 [0.650, 0.656]"
6,casualty_count_removed,lr,0.30,"0.445 [0.439, 0.451]","0.724 [0.720, 0.727]","0.056 [0.042, 0.071]","0.639 [0.636, 0.642]"
7,casualty_count_removed,rf,0.30,"0.457 [0.451, 0.463]","0.732 [0.729, 0.736]","0.035 [0.021, 0.050]","0.644 [0.641, 0.647]"
8,casualty_count_removed,hgb,0.31,"0.464 [0.458, 0.470]","0.737 [0.733, 0.740]","0.048 [0.034, 0.063]","0.648 [0.645, 0.651]"



Variant minus frozen baseline, same collision draws (95% interval):


,variant,model,measure,difference,lo,hi,interval_excludes_zero
0,force_included,lr,ap,0.0034,-0.0009,0.0072,False
1,force_included,lr,auc,0.0044,0.0033,0.0055,True
3,force_included,lr,cal_intercept,0.0442,0.0414,0.0466,True
5,force_included,lr,macro_f1 (own tuned threshold),0.0029,0.0009,0.0048,True
9,force_included,rf,ap,0.0036,0.0021,0.0050,True
10,force_included,rf,auc,0.0034,0.0025,0.0043,True
12,force_included,rf,cal_intercept,0.0072,0.0059,0.0085,True
14,force_included,rf,macro_f1 (own tuned threshold),0.0014,-0.0003,0.0029,False
18,force_included,hgb,ap,0.0067,0.0050,0.0083,True
19,force_included,hgb,auc,0.0047,0.0036,0.0058,True



Calibration intercept gap between stable injury-based and stable non-injury-based forces (baseline gap from notebook 07 for hgb was 0.19):


,variant,model,intercept_gap_injury_minus_non_injury,lo,hi
0,base,lr,0.2393,0.1391,0.3653
1,base,rf,0.2444,0.1802,0.3061
2,base,hgb,0.1922,0.1276,0.2536
3,force_included,lr,0.0699,-0.0472,0.2350
4,force_included,rf,0.1760,0.1126,0.2368
5,force_included,hgb,0.0501,-0.0130,0.1113
6,casualty_count_removed,lr,0.1855,0.1193,0.2471
7,casualty_count_removed,rf,0.2505,0.1874,0.3116
8,casualty_count_removed,hgb,0.1869,0.1233,0.2494



Model ordering inside each variant:


,variant,measure,point_order,share_of_draws_hgb_first
0,base,ap,hgb > rf > lr,1.0
1,base,auc,hgb > rf > lr,1.0
2,force_included,ap,hgb > rf > lr,1.0
3,force_included,auc,hgb > rf > lr,1.0
4,casualty_count_removed,ap,hgb > rf > lr,1.0
5,casualty_count_removed,auc,hgb > rf > lr,1.0


In [7]:
# 7. Log the departure and record what was done -----------------------------------------------------------------------------------
dep_path = FINAL / 'departures_log.json'
log = json.loads(dep_path.read_text()) if dep_path.exists() else []
if not any(d.get('id') == 'sensitivity_untreated_only' for d in log):
    log.append(dict(id='sensitivity_untreated_only', logged=dt.datetime.now().isoformat(timespec='seconds'), notebook='08',
                    plan='Chapter 3: sensitivity analyses for all nine model and training combinations.',
                    done='Sensitivity refits (police force added; number of casualties removed) run for the three untreated models only.',
                    reason='Compute. Training treatments did not improve ranking in notebooks 06 and 07, and the selected model is untreated.',
                    approved_by='Kojo (user), 2026-11-07'))
    dep_path.write_text(json.dumps(log, indent=1))
files = sorted(p.name for p in RES.glob('*') if p.name.startswith(('temporal_', 'sensitivity_', 'bootstrap_draws_temporal', 'bootstrap_draws_sensitivity')))
summary = dict(finished=dt.datetime.now().isoformat(timespec='seconds'), notebook='08', quick_test=QUICK_TEST, bootstrap_replicates=REPS,
               seeds=dict(folds_2023=SEED + 5, bootstrap_2024=SEED + 6, bootstrap_sensitivity=SEED + 7), parts_run=dict(A=RUN_A, B=RUN_B),
               folds_2023_sha256=sha(SENS / 'folds_2023_only.csv') if (SENS / 'folds_2023_only.csv').exists() else None,
               frozen_settings=SET, variants={k: dict(numeric=len(v['num']), categorical=len(v['cat'])) for k, v in VARIANTS.items()},
               departures=[d['id'] for d in json.loads(dep_path.read_text())], outputs=files,
               output_sha256={f: sha(RES / f) for f in files if not f.startswith('bootstrap_draws')})
(RES / 'results_notebook08_summary.json').write_text(json.dumps(summary, indent=1))
print(json.dumps({k: v for k, v in summary.items() if k != 'output_sha256'}, indent=1))
print('\nNEXT: notebook 09 runs permutation importance for the selected model.')

{
 "finished": "2026-11-07T21:55:02",
 "notebook": "08",
 "quick_test": false,
 "bootstrap_replicates": 2000,
 "seeds": {
  "folds_2023": 20261112,
  "bootstrap_2024": 20261113,
  "bootstrap_sensitivity": 20261114
 },
 "parts_run": {
  "A": true,
  "B": true
 },
 "folds_2023_sha256": "f895e27cdf64a5eb20d8b96023674f7c996639f4db04fe58a0aef0d1c5c37507",
 "frozen_settings": {
  "lr": {
   "C": 0.1
  },
  "rf": {
   "max_features": 0.2,
   "min_samples_leaf": 10,
   "min_samples_split": 10
  },
  "hgb": {
   "learning_rate": 0.05,
   "max_iter": 200,
   "max_leaf_nodes": 63,
   "min_samples_leaf": 50,
   "l2_regularization": 0.0
  }
 },
 "variants": {
  "force_included": {
   "numeric": 8,
   "categorical": 40
  },
  "casualty_count_removed": {
   "numeric": 7,
   "categorical": 39
  }
 },
 "departures": [
  "sensitivity_untreated_only"
 ],
 "outputs": [
  "bootstrap_draws_sensitivity_2025.parquet",
  "bootstrap_draws_temporal_2024.parquet",
  "sensitivity_2025_by_recording_system.csv",
  "